In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re
import os
import pandas as pd

d:\Program\envs\agent_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
def read_markdown_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def clean_markdown_headers(text):
    # Xóa các dấu ** bao quanh text nằm ngay sau các dấu #
    return re.sub(r'(#+)\s*\*\*(.*?)\*\* ', r'\1 \2', text)


In [3]:
configs = {
    "Guidelines.md": {
        "document_type": "guideline",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II"),
            ("###", "Heading_III")
        ]
    },
    "Monographs.md": {
        "document_type": "monograph",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II")
        ]
    },
    "Appendices_2.md": {
        "document_type": "appendix",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II")
        ]
    },
    "Appendices_3.md": {
        "document_type": "appendix",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II"),
            ("###", "Heading_III")
        ]
    }
}

In [4]:
MAX_TOKENS = 8191 - 200
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="text-embedding-3-large",
    chunk_size=MAX_TOKENS,
    chunk_overlap=0,
)


In [5]:
def process_file_to_chunks(file_path, config):
    raw_text = read_markdown_file(file_path)
    cleaned_text = clean_markdown_headers(raw_text)
    
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=config["headers_to_split_on"],
        strip_headers=True
    )
    md_header_splits = markdown_splitter.split_text(cleaned_text)
    final_splits = text_splitter.split_documents(md_header_splits)
    
    data_to_save = []
    for chunk in final_splits:
        row = {
            "document_type": config["document_type"],
            "level_1": chunk.metadata.get("Heading_I", None),
            "level_2": chunk.metadata.get("Heading_II", None),
            "level_3": chunk.metadata.get("Heading_III", None),
            "content": chunk.page_content
        }
        data_to_save.append(row)
        
    return data_to_save

In [6]:

input_dir = r"D:\PharmaRAG-VN\Data\Clean"
output_dir = r"D:\PharmaRAG-VN\Data\Clean"

# 1. Khởi tạo một list tổng để chứa tất cả các chunk từ mọi file
all_chunks = []

for filename, config in configs.items():
    file_path = os.path.join(input_dir, filename)
    if os.path.exists(file_path):
        print(f"--- Processing {filename} ---")
        chunks = process_file_to_chunks(file_path, config)
        
        # 2. Đưa dữ liệu vừa xử lý vào list tổng
        all_chunks.extend(chunks)
        print(f"Processed {len(chunks)} chunks.\n")
    else:
        print(f"File not found: {file_path}\n")

# 3. Sau khi chạy xong toàn bộ file, gom vào DataFrame và lưu ra 1 file CSV
if all_chunks:
    print("--- Saving All Data ---")
    df = pd.DataFrame(all_chunks)
    df = df[["document_type", "level_1", "level_2", "level_3", "content"]]
    
    output_filename = "All_Documents_chunk.csv" # Đặt tên cho file tổng
    output_path = os.path.join(output_dir, output_filename)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ Successfully saved {len(all_chunks)} total chunks to: {output_path}\n")
else:
    print("Không có dữ liệu nào được xử lý thành công để lưu.")


--- Processing Guidelines.md ---
Processed 113 chunks.

--- Processing Monographs.md ---
Processed 6715 chunks.

--- Processing Appendices_2.md ---
Processed 10 chunks.

--- Processing Appendices_3.md ---
Processed 18 chunks.

--- Saving All Data ---
✅ Successfully saved 6856 total chunks to: D:\PharmaRAG-VN\Data\Clean\All_Documents_chunk.csv

